# Visao Geral dos Resultados Atlas

Este notebook resume os graficos e tabelas produzidos na pasta `reports` e explica em linguagem simples o que cada saida significa. Use-o como guia rapido para apresentar os resultados a quem nao tem familiaridade com financas quantitativas.


## Como usar este material

1. Rode as celulas em ordem usando Jupyter.
2. Cada secao mostra um grafico ou tabela seguido de um texto curto que traduz o resultado para termos do dia a dia.
3. Sempre que voce gerar novos artefatos na pasta `reports`, reexecute as celulas para atualizar as figuras.


In [ ]:
from pathlib import Path
import ast
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

plt.style.use('seaborn-v0_8')

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "reports").exists():
    REPORTS_DIR = NOTEBOOK_DIR / "reports"
else:
    REPORTS_DIR = NOTEBOOK_DIR.parent / "reports"
REPORTS_DIR = REPORTS_DIR.resolve()
print(f"Usando pasta de reports: {REPORTS_DIR}")


## 1. Backtest base

O backtest mostra como a estrategia teria se comportado usando todo o historico disponivel, sem re-treinos fora da amostra. Pense nele como um replay do passado sob as regras do modelo.


In [ ]:
equity_candidates = sorted(REPORTS_DIR.glob('equity_curve*.csv'), key=lambda p: p.stat().st_mtime)
if not equity_candidates:
    raise FileNotFoundError('Nenhum equity_curve*.csv encontrado em reports')
equity_path = equity_candidates[-1]
print(f'Fonte: {equity_path.name}')

equity_df = pd.read_csv(equity_path, parse_dates=['date'])
equity_series = equity_df.set_index('date').iloc[:, 0].rename('equity')

fig, ax = plt.subplots(figsize=(10, 4))
equity_series.plot(ax=ax, color='#1f77b4')
ax.axhline(0.0, color='black', linewidth=0.8, alpha=0.3)
ax.set_title('Evolucao diaria da carteira no backtest')
ax.set_ylabel('Equity normalizada')
ax.set_xlabel('Data')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

resumo_backtest = equity_series.describe().to_frame('valor')
display(resumo_backtest)


**Como interpretar:** valores negativos indicam que a carteira perdeu dinheiro ao longo do periodo. Como a linha ficou quase flat em torno de zero, significa que a estrategia nao gerou ganhos sustentaveis ate aqui. Esse diagnostico sugere revisar fatores, custos ou regras de gestao de risco.


## 2. Walk-forward (simulacao fora da amostra)

O walk-forward alterna janelas de treino e teste para medir o desempenho quando o modelo precisa se adaptar com informacoes novas, simulando uma rotina operacional real.


In [ ]:
walk_path = REPORTS_DIR / "walkforward_equity.csv"
if walk_path.exists():
    walk_df = pd.read_csv(walk_path, parse_dates=['date'])
    walk_series = walk_df.set_index('date').iloc[:, 0].rename('equity_walk')
    fig, ax = plt.subplots(figsize=(10, 4))
    walk_series.plot(ax=ax, color='#ff7f0e')
    ax.axhline(1.0, color='black', linewidth=0.8, alpha=0.3)
    ax.set_title('Evolucao da carteira em modo walk-forward')
    ax.set_ylabel('Equity acumulada')
    ax.set_xlabel('Data')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    resumo_wf = walk_series.describe().to_frame('valor')
    display(resumo_wf)
else:
    print('Arquivo walkforward_equity.csv nao encontrado em reports.')


**Como interpretar:** o grafico mostra o capital acumulado quando repetimos o processo com reequilibrios e re-treinos periodicos. Se a linha cai, quer dizer que o modelo perde dinheiro quando sai do laboratorio. O objetivo eh que ela suba de forma suave; caso contrario, precisamos investigar causas (fatores instaveis, custos altos ou universo inadequado).


## 3. KPIs do backtest agregados

Aqui vemos as metricas classicas (Sharpe, volatilidade, drawdown). Mesmo sem entender formulas, basta olhar os sinais: positivo e alto eh bom; negativo ou muito baixo eh um alerta.


In [ ]:
kpi_path = REPORTS_DIR / "kpi_table.csv"
if kpi_path.exists():
    kpi_df = pd.read_csv(kpi_path)
    display(kpi_df)
else:
    print('Arquivo kpi_table.csv nao encontrado em reports.')


**Como interpretar:**
- `Sharpe` negativo indica que o risco tomado nao foi compensado por retorno.
- `Vol` mostra quanto a carteira oscila; quanto maior, mais montanha-russa.
- `MaxDD` eh o pior tombo observado. Valores muito baixos (perto de -1) significam queda quase total.


## 4. Ajuste fino de parametros (tuning)

A tabela abaixo lista combinacoes testadas para os parametros topologicos e o Sharpe correspondente. Ela ajuda a descobrir se existe algum conjunto que funcione melhor ou se o modelo e instavel.


In [ ]:
tuning_path = REPORTS_DIR / "tuning_results.csv"
if tuning_path.exists():
    tuning_df = pd.read_csv(tuning_path)
    if "params" in tuning_df.columns:
        try:
            tuning_df["params"] = tuning_df["params"].apply(ast.literal_eval)
        except (ValueError, SyntaxError):
            pass
    tuning_sorted = tuning_df.sort_values(by="sharpe", ascending=False)
    display(tuning_sorted.head(10))
else:
    print('Arquivo tuning_results.csv nao encontrado em reports.')


**Como interpretar:** procure linhas com Sharpe positivo. Se todas as combinacoes ficam negativas, e sinal de que precisamos repensar como os fatores e o grafo TDA estao sendo calculados.


## 5. Sensibilidade dos parametros

Os mapas de calor comparam pares de parametros e usam cores para indicar desempenho. Eles sao uteis para explicar se o modelo eh robusto (area grande com tons positivos) ou se so funciona em um ponto especifico.


In [ ]:
for title, pattern in [('Sharpe', "heatmap_sharpe_*.png"), ('Volatilidade', "heatmap_vol_*.png")]:
    candidates = sorted(REPORTS_DIR.glob(pattern), key=lambda p: p.stat().st_mtime)
    if candidates:
        img_path = candidates[-1]
        print(f"{title}: {img_path.name}")
        display(Image(filename=str(img_path)))
    else:
        print(f"Nenhum arquivo encontrado para {pattern}")


**Como interpretar:** cores mais quentes (laranja/vermelho) apontam combinacoes com desempenho pior; tons frios (azul) indicam configuracoes relativamente melhores. Buscamos padroes coerentes, nao pontos isolados.


## 6. Capacidade operacional (capacity curve)

Esta analise projeta o impacto de aumentar o tamanho das ordens sobre o resultado. Ela responde se o modelo suporta mais capital sem degradar a performance.


In [ ]:
capacity_csv = sorted(REPORTS_DIR.glob('capacity_curve_*.csv'), key=lambda p: p.stat().st_mtime)
capacity_png = sorted(REPORTS_DIR.glob('capacity_curve_*.png'), key=lambda p: p.stat().st_mtime)
if capacity_csv:
    cap_path = capacity_csv[-1]
    cap_df = pd.read_csv(cap_path)
    print(f'Tabela: {cap_path.name}')
    display(cap_df)
else:
    print('Nenhum capacity_curve_*.csv encontrado.')

if capacity_png:
    cap_img = capacity_png[-1]
    print(f'Grafico: {cap_img.name}')
    display(Image(filename=str(cap_img)))
else:
    print('Nenhum capacity_curve_*.png encontrado.')


**Como interpretar:** a tabela mostra como Sharpe e drawdown mudam quando limitamos a participacao (porcentagem do volume diario). Se os numeros pioram muito ao aumentar o limite, significa que a estrategia so funciona com ordens pequenas.


## 7. Stress de custos

Aqui simulamos cenarios com custos de transacao mais altos ou mais baixos para ver o quao sensivel o modelo e a essas variacoes.


In [ ]:
stress_csv = sorted(REPORTS_DIR.glob('stress_costs_*.csv'), key=lambda p: p.stat().st_mtime)
if stress_csv:
    stress_path = stress_csv[-1]
    stress_df = pd.read_csv(stress_path)
    print(f'Fonte: {stress_path.name}')
    display(stress_df)
else:
    print('Nenhum stress_costs_*.csv encontrado.')


**Como interpretar:** se um pequeno aumento no multiplicador de custo derruba muito os retornos, a estrategia pode nao sobreviver em mercados menos liquidos ou com corretagem maior. Buscamos colunas que mudem pouco conforme o custo sobe.


## 8. Proximos passos sugeridos

- Ajustar fatores, universo ou parametros para buscar Sharpe positivo e drawdowns menores.
- Repetir os testes sempre que novos dados forem incorporados.
- Usar este notebook como material de apresentacao para areas nao tecnicas, mantendo o foco nas mensagens principais: estabilidade, consistencia e viabilidade operacional.
